In [9]:
import geopandas as gpd
import pandas as pd
import fiona

# Habilitar el soporte de lectura KML en fiona (necesario para geopandas)
fiona.drvsupport.supported_drivers['KML'] = 'rw'

In [10]:
# Apunta al archivo .kml que extrajiste del KMZ
ruta_kml = 'Derrame de Petróleo Veracruz-Tabasco 2026.kml'

In [11]:
# 1. Listar todas las capas (layers) que hay dentro del archivo
capas = fiona.listlayers(ruta_kml)
print(f"Se encontraron estas capas en el archivo: {capas}")

Se encontraron estas capas en el archivo: ['Reportes ciudadanos chapopote', 'Reportes ciudadanos flora y fauna con posibles afectaciones', 'Arrecifes CASGM', 'Pozos petroleros Aguas Someras', 'Pozos Petroleros Aguas Profundas', 'Área Corredor Arrecifal (CASGM)']


In [13]:
# 2. Iterar sobre las capas para encontrar la que tiene los datos
dataframes = []
for capa in capas:
    # Leemos capa por capa
    gdf_temp = gpd.read_file(ruta_kml, driver='KML', layer=capa)
    
    # Si la capa no está vacía, la guardamos
    if not gdf_temp.empty:
        print(f"¡Datos encontrados en la capa: '{capa}' con {len(gdf_temp)} registros!")
        dataframes.append(gdf_temp)

¡Datos encontrados en la capa: 'Reportes ciudadanos chapopote' con 106 registros!
¡Datos encontrados en la capa: 'Reportes ciudadanos flora y fauna con posibles afectaciones' con 20 registros!
¡Datos encontrados en la capa: 'Arrecifes CASGM' con 125 registros!
¡Datos encontrados en la capa: 'Pozos petroleros Aguas Someras' con 2000 registros!
¡Datos encontrados en la capa: 'Pozos Petroleros Aguas Profundas' con 106 registros!
¡Datos encontrados en la capa: 'Área Corredor Arrecifal (CASGM)' con 1 registros!


In [43]:
# 1. CARGAR LOS DATOS (La orden real)
# Aquí usamos gpd.read_file para darle la orden a la computadora de abrir el archivo KML 
# y extraer específicamente la capa que queremos. Lo guardamos en una variable llamada 'gdf'.
gdf = gpd.read_file(ruta_kml, driver='KML', layer='Reportes ciudadanos chapopote')

# 2. LIMPIAR LOS DATOS
# Le decimos a Python: "De la tabla 'gdf', consérvame solo las filas que tengan un punto en el mapa válido".
gdf = gdf[gdf.geometry.is_valid]

# Le decimos: "Si hay alguna fila que esté completamente vacía en su columna de mapa (geometry), bórrala".
gdf = gdf.dropna(subset=['geometry'])

# 3. MOSTRAR EN PANTALLA (El paso que faltaba)
# El comando display() o print() es como decirle a Python: "Pon esta tabla sobre mi escritorio para verla".
# .head() le dice que solo te muestre las primeras 5 filas para no saturar tu pantalla.
display(gdf.head())

,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,Descripci__n,SITIO_OBS,MUN,ESTADO,Limpieza,FechaRepor,gx_media_links,geometry
0,28,Barillas,"<img src=""https://mymaps.usercontent.google.co...",NaT,NaT,NaT,None,-1.0,0.0,-1,NaN,None,03-03-2026 Primer reporte de chapopote el 3 de...,Barrillas,Coatzacoalcos,Veracruz,Autoridades-Comunidad,03-03-2026,https://mymaps.usercontent.google.com/hostedim...,POINT Z (-94.59051 18.18613 0)
1,24,Paquital,Descripción: <br>id: 24<br>SITIO_OBS: Paquital...,NaT,NaT,NaT,None,-1.0,0.0,-1,NaN,None,,Paquital,Pajapan,Veracruz,Comunidad,03-03-2026,NaN,POINT Z (-94.64814 18.18149 0)
2,22,Playa Linda,"<img src=""https://mymaps.usercontent.google.co...",NaT,NaT,NaT,None,-1.0,0.0,-1,NaN,None,Primer reporte de chapopote el 3 de marzo. 7 d...,Playa Linda,Pajapan,Veracruz,Autoridades-Comunidad,03-03-2026,https://mymaps.usercontent.google.com/hostedim...,POINT Z (-94.62356 18.25027 0)
3,17,Peña Hermosa,"<img src=""https://mymaps.usercontent.google.co...",NaT,NaT,NaT,None,-1.0,0.0,-1,NaN,None,Presencia de chapopote en la parte de la playa...,Peña Hermosa,Tatahuicapan De J.,Veracruz,Ninguna,03-03-2026,https://mymaps.usercontent.google.com/hostedim...,POINT Z (-94.68383 18.35097 0)
4,7,Los Arrecifes,"<img src=""https://mymaps.usercontent.google.co...",NaT,NaT,NaT,None,-1.0,0.0,-1,NaN,None,02-03-2026 Primer reporte (foto). \n07-03-2026...,Los Arrecifes,Mecayacapan,Veracruz,Ninguna,02-03-2026,https://mymaps.usercontent.google.com/hostedim...,POINT Z (-94.8512 18.54442 0)


In [44]:
import pandas as pd

print("1. Extrayendo coordenadas del mapa...")
# Sacamos el eje Y (Latitud) y el eje X (Longitud) de la columna 'geometry'
gdf['Latitud'] = gdf.geometry.y
gdf['Longitud'] = gdf.geometry.x

print("2. Convirtiendo el mapa a una tabla normal...")
# Creamos un DataFrame de Pandas normal y borramos la columna espacial 'geometry'
# porque un archivo CSV estándar no la entiende.
df_limpio = pd.DataFrame(gdf.drop(columns=['geometry']))

print("3. Guardando el archivo en tu computadora...")
# Guardamos la tabla como un archivo .csv
# index=False evita que se guarde una columna extra con números de fila (0, 1, 2...)
df_limpio.to_csv('reportes_chapopote_limpios.csv', index=False, encoding='utf-8-sig')

print("\n¡Proceso Terminado!")
print("Si vas a la carpeta donde está tu Jupyter Notebook, verás un nuevo archivo llamado 'reportes_chapopote_limpios.csv'.")

# Damos un último vistazo a cómo quedó la tabla final
display(df_limpio.head())

1. Extrayendo coordenadas del mapa...
2. Convirtiendo el mapa a una tabla normal...
3. Guardando el archivo en tu computadora...

¡Proceso Terminado!
Si vas a la carpeta donde está tu Jupyter Notebook, verás un nuevo archivo llamado 'reportes_chapopote_limpios.csv'.


,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,...,icon,Descripci__n,SITIO_OBS,MUN,ESTADO,Limpieza,FechaRepor,gx_media_links,Latitud,Longitud
0,28,Barillas,"<img src=""https://mymaps.usercontent.google.co...",NaT,NaT,NaT,None,-1.0,0.0,-1,...,None,03-03-2026 Primer reporte de chapopote el 3 de...,Barrillas,Coatzacoalcos,Veracruz,Autoridades-Comunidad,03-03-2026,https://mymaps.usercontent.google.com/hostedim...,18.186126,-94.590514
1,24,Paquital,Descripción: <br>id: 24<br>SITIO_OBS: Paquital...,NaT,NaT,NaT,None,-1.0,0.0,-1,...,None,,Paquital,Pajapan,Veracruz,Comunidad,03-03-2026,NaN,18.181494,-94.648139
2,22,Playa Linda,"<img src=""https://mymaps.usercontent.google.co...",NaT,NaT,NaT,None,-1.0,0.0,-1,...,None,Primer reporte de chapopote el 3 de marzo. 7 d...,Playa Linda,Pajapan,Veracruz,Autoridades-Comunidad,03-03-2026,https://mymaps.usercontent.google.com/hostedim...,18.250268,-94.623562
3,17,Peña Hermosa,"<img src=""https://mymaps.usercontent.google.co...",NaT,NaT,NaT,None,-1.0,0.0,-1,...,None,Presencia de chapopote en la parte de la playa...,Peña Hermosa,Tatahuicapan De J.,Veracruz,Ninguna,03-03-2026,https://mymaps.usercontent.google.com/hostedim...,18.350966,-94.683833
4,7,Los Arrecifes,"<img src=""https://mymaps.usercontent.google.co...",NaT,NaT,NaT,None,-1.0,0.0,-1,...,None,02-03-2026 Primer reporte (foto). \n07-03-2026...,Los Arrecifes,Mecayacapan,Veracruz,Ninguna,02-03-2026,https://mymaps.usercontent.google.com/hostedim...,18.544423,-94.851204
